# 2.6 Improving Training with Metrics and Data Augmentation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emreaslan7/ai/blob/main/notebooks/deep-learning-with-pytorch/14-improving-training-with-metrics-and-augmentation.ipynb)

This notebook implements Section 2.6: Resolving the Accuracy Paradox in 3D lung tumor classification through clinical metric tracking, balanced stratified dataset sampling, and 3D volumetric data augmentation.

In [ ]:
# Cell 0: Core Setup, Imports & Device Verification
import math
import random
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import SGD
from torch.utils.data import DataLoader, Dataset

# Set seeds for reproducible execution
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__} | Active Compute Device: {device}")

## 1. 3D Volumetric Candidate Patch Dataset with Dynamic Balancing

We implement a synthetic 3D CT patch generator ($32 \times 32 \times 32$ voxels) that can operate either in raw imbalanced mode ($99\%$ negative) or dynamically balanced mode (`ratio_int=1`).

In [ ]:
class SyntheticLunaDataset(Dataset):
    """Generates synthetic 3D CT candidate sub-volumes with optional class balancing and augmentation."""
    def __init__(self, num_samples=1000, positive_ratio=0.01, ratio_int=0, augment_bool=False, is_val=False):
        self.num_samples = num_samples
        self.ratio_int = ratio_int
        self.augment_bool = augment_bool
        self.is_val = is_val
        
        num_positives = max(10, int(num_samples * positive_ratio))
        self.pos_indices = list(range(num_positives))
        self.neg_indices = list(range(num_positives, num_samples))
        
    def __len__(self):
        if self.ratio_int > 0 and not self.is_val:
            return len(self.pos_indices) * (self.ratio_int + 1)
        return self.num_samples

    def _generate_patch(self, is_nodule):
        # Lung parenchyma background noise
        patch = torch.randn(1, 32, 32, 32) * 0.2
        if is_nodule:
            # Soft spherical nodule at center (radius ~ 5 voxels)
            coords = torch.stack(torch.meshgrid(
                torch.arange(32) - 16,
                torch.arange(32) - 16,
                torch.arange(32) - 16,
                indexing="ij"
            ))
            radius_sq = (coords ** 2).sum(dim=0).float()
            patch[0, radius_sq <= 25.0] += 0.8
        return patch.clamp_(-1.0, 1.0)

    def __getitem__(self, ndx):
        if self.ratio_int > 0 and not self.is_val:
            if ndx % (self.ratio_int + 1) == 0:
                pos_ndx = (ndx // (self.ratio_int + 1)) % len(self.pos_indices)
                is_nodule = 1
            else:
                neg_ndx = ndx % len(self.neg_indices)
                is_nodule = 0
        else:
            is_nodule = 1 if ndx in self.pos_indices else 0
            
        patch = self._generate_patch(is_nodule)
        
        if self.augment_bool:
            patch = apply_3d_augmentation(patch)
            
        label = torch.tensor(is_nodule, dtype=torch.long)
        return patch, label

## 2. 3D Volumetric Affine Data Augmentation Pipeline

We construct random $3 \times 4$ affine transformation matrices combining 3D rotations, flips, spatial scaling, and translations, resampling voxels using `torch.nn.functional.affine_grid` and `torch.nn.functional.grid_sample`.

In [ ]:
def build_3d_affine_matrix():
    """Generates a random 3x4 affine matrix for 3D rotation, scaling, and translation."""
    # 1. Random 3D flips
    fd = -1.0 if random.random() > 0.5 else 1.0
    fh = -1.0 if random.random() > 0.5 else 1.0
    fw = -1.0 if random.random() > 0.5 else 1.0
    
    # 2. Random 3D scaling (0.85 to 1.15)
    scale = random.uniform(0.85, 1.15)
    
    # 3. Random 3D Euler angles
    az = random.uniform(-math.pi, math.pi)
    ay = random.uniform(-math.pi / 4, math.pi / 4)
    ax = random.uniform(-math.pi / 4, math.pi / 4)
    
    cz, sz = math.cos(az), math.sin(az)
    cy, sy = math.cos(ay), math.sin(ay)
    cx, sx = math.cos(ax), math.sin(ax)
    
    Rz = torch.tensor([[cz, -sz, 0., 0.], [sz, cz, 0., 0.], [0., 0., 1., 0.], [0., 0., 0., 1.]])
    Ry = torch.tensor([[cy, 0., sy, 0.], [0., 1., 0., 0.], [-sy, 0., cy, 0.], [0., 0., 0., 1.]])
    Rx = torch.tensor([[1., 0., 0., 0.], [0., cx, -sx, 0.], [0., sx, cx, 0.], [0., 0., 0., 1.]])
    
    S = torch.tensor([
        [scale * fd, 0., 0., 0.],
        [0., scale * fh, 0., 0.],
        [0., 0., scale * fw, 0.],
        [0., 0., 0., 1.]
    ])
    
    T = torch.tensor([
        [1., 0., 0., random.uniform(-0.05, 0.05)],
        [0., 1., 0., random.uniform(-0.05, 0.05)],
        [0., 0., 1., random.uniform(-0.05, 0.05)],
        [0., 0., 0., 1.]
    ])
    
    affine_4x4 = T @ Rz @ Ry @ Rx @ S
    return affine_4x4[:3] # (3, 4)

def apply_3d_augmentation(patch):
    """Applies GPU-accelerated trilinear 3D resampling to a (1, D, H, W) patch."""
    input_t = patch.unsqueeze(0) # (1, 1, 32, 32, 32)
    affine_3x4 = build_3d_affine_matrix().unsqueeze(0) # (1, 3, 4)
    
    grid = F.affine_grid(affine_3x4, input_t.size(), align_corners=False)
    augmented = F.grid_sample(input_t, grid, mode='bilinear', padding_mode='border', align_corners=False)
    
    noise = torch.randn_like(augmented) * 0.02
    return (augmented + noise).squeeze(0).clamp_(-1.0, 1.0)

## 3. Visualizing 3D Augmentation Transforms Across Orthogonal Planes

We inspect intermediate axial, coronal, and sagittal slice views before and after 3D affine transformation.

In [ ]:
raw_dataset = SyntheticLunaDataset(num_samples=100, positive_ratio=1.0, augment_bool=False)
raw_patch, _ = raw_dataset[0]
aug_patch = apply_3d_augmentation(raw_patch)

fig, axes = plt.subplots(2, 3, figsize=(10, 6))
planes = [("Axial (D=16)", 16, 0), ("Coronal (H=16)", 16, 1), ("Sagittal (W=16)", 16, 2)]

for col, (title, slice_idx, axis) in enumerate(planes):
    if axis == 0:
        r_slice = raw_patch[0, slice_idx, :, :].numpy()
        a_slice = aug_patch[0, slice_idx, :, :].numpy()
    elif axis == 1:
        r_slice = raw_patch[0, :, slice_idx, :].numpy()
        a_slice = aug_patch[0, :, slice_idx, :].numpy()
    else:
        r_slice = raw_patch[0, :, :, slice_idx].numpy()
        a_slice = aug_patch[0, :, :, slice_idx].numpy()
        
    axes[0, col].imshow(r_slice, cmap="gray", vmin=-1.0, vmax=1.0)
    axes[0, col].set_title(f"Raw: {title}")
    axes[0, col].axis("off")
    
    axes[1, col].imshow(a_slice, cmap="gray", vmin=-1.0, vmax=1.0)
    axes[1, col].set_title(f"Augmented: {title}")
    axes[1, col].axis("off")

plt.tight_layout()
plt.show()

## 4. 3D CNN Classifier (`LunaModel`)

Hierarchical 4-stage 3D convolution backbone with spatial max pooling and classification head.

In [ ]:
class LunaBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv1 = nn.Conv3d(in_c, out_c, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv3d(out_c, out_c, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)

    def forward(self, x):
        x = self.relu1(self.conv1(x))
        x = self.relu2(self.conv2(x))
        return self.pool(x)

class LunaModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.tail_bn = nn.BatchNorm3d(1)
        self.block1 = LunaBlock(1, 8)
        self.block2 = LunaBlock(8, 16)
        self.block3 = LunaBlock(16, 32)
        self.block4 = LunaBlock(32, 64)
        self.fc = nn.Linear(64 * 2 * 2 * 2, 2)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.tail_bn(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = x.view(x.size(0), -1)
        logits = self.fc(x)
        probs = self.softmax(logits)
        return logits, probs

model = LunaModel().to(device)
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"LunaModel Initialized | Trainable Parameters: {param_count:,}")

## 5. Clinical Metrics Engine

Computes Sensitivity (Recall), Precision, Specificity, and the harmonic $F_1$-score from the aggregated Confusion Matrix.

In [ ]:
def compute_metrics(y_true, y_pred, threshold=0.5):
    pred_pos = (y_pred >= threshold)
    pred_neg = (y_pred < threshold)
    true_pos = (y_true == 1)
    true_neg = (y_true == 0)
    
    tp = (pred_pos & true_pos).sum().item()
    fp = (pred_pos & true_neg).sum().item()
    tn = (pred_neg & true_neg).sum().item()
    fn = (pred_neg & true_pos).sum().item()
    
    recall = tp / (tp + fn + 1e-8)
    precision = tp / (tp + fp + 1e-8)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-8)
    accuracy = (tp + tn) / (tp + tn + fp + fn + 1e-8)
    
    return {"TP": tp, "FP": fp, "TN": tn, "FN": fn, "Recall": recall, "Precision": precision, "F1": f1, "Accuracy": accuracy}

## 6. Training Execution: Comparing Baseline vs. Balanced + Augmented

We train two identical networks for 5 epochs: Model A without balancing/augmentation (suffering the accuracy paradox), and Model B with dynamic 1:1 balancing and 3D affine data augmentation.

In [ ]:
def train_experiment(name, ratio_int, augment_bool, epochs=5):
    print(f"\n=== Starting Experiment: {name} (ratio_int={ratio_int}, augment={augment_bool}) ===")
    train_ds = SyntheticLunaDataset(num_samples=800, positive_ratio=0.02, ratio_int=ratio_int, augment_bool=augment_bool, is_val=False)
    val_ds = SyntheticLunaDataset(num_samples=200, positive_ratio=0.02, ratio_int=0, augment_bool=False, is_val=True)
    
    train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=32, shuffle=False)
    
    exp_model = LunaModel().to(device)
    optimizer = SGD(exp_model.parameters(), lr=0.005, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    
    history = {"val_loss": [], "val_recall": [], "val_precision": [], "val_f1": []}
    
    for epoch in range(1, epochs + 1):
        exp_model.train()
        for x_b, y_b in train_dl:
            x_b, y_b = x_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            logits, _ = exp_model(x_b)
            loss = criterion(logits, y_b)
            loss.backward()
            optimizer.step()
            
        # Validation
        exp_model.eval()
        val_loss, all_preds, all_labels = 0.0, [], []
        with torch.inference_mode():
            for x_b, y_b in val_dl:
                x_b, y_b = x_b.to(device), y_b.to(device)
                logits, probs = exp_model(x_b)
                val_loss += criterion(logits, y_b).item() * len(y_b)
                all_preds.extend(probs[:, 1].cpu().tolist())
                all_labels.extend(y_b.cpu().tolist())
                
        val_loss /= len(val_ds)
        m = compute_metrics(torch.tensor(all_labels), torch.tensor(all_preds))
        history["val_loss"].append(val_loss)
        history["val_recall"].append(m["Recall"])
        history["val_precision"].append(m["Precision"])
        history["val_f1"].append(m["F1"])
        
        print(f"Epoch {epoch:2d} | Val Loss: {val_loss:.4f} | Acc: {m['Accuracy']*100:.1f}% | Recall: {m['Recall']*100:.1f}% | Prec: {m['Precision']*100:.1f}% | F1: {m['F1']:.4f}")
        
    return history

hist_baseline = train_experiment("Baseline (Unbalanced, No Aug)", ratio_int=0, augment_bool=False)
hist_augmented = train_experiment("Balanced + 3D Augmentation", ratio_int=1, augment_bool=True)

## 7. Comparative Metric Dynamics: The Breakthrough

We graph the validation loss and $F_1$-score progression to visualize how balanced sampling and 3D data augmentation overcome the accuracy paradox.

In [ ]:
epochs_range = range(1, 6)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Loss curves
ax1.plot(epochs_range, hist_baseline["val_loss"], 'r--o', label="Baseline Validation Loss")
ax1.plot(epochs_range, hist_augmented["val_loss"], 'g-s', label="Augmented Validation Loss")
ax1.set_title("Validation Loss Comparison")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross Entropy Loss")
ax1.grid(True, linestyle="--", alpha=0.6)
ax1.legend()

# F1 Score comparison
ax2.plot(epochs_range, hist_baseline["val_f1"], 'r--o', label="Baseline F1 Score")
ax2.plot(epochs_range, hist_augmented["val_f1"], 'g-s', label="Augmented F1 Score")
ax2.set_title("Validation F1-Score Progression")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Harmonic F1 Score")
ax2.grid(True, linestyle="--", alpha=0.6)
ax2.legend()

plt.tight_layout()
plt.show()